In [0]:
BOOTSTRAP_SERVER = "d868cd8hvfrjvm53hr40.any.ap-south-1.mpx.prd.cloud.redpanda.com:9092"
USERNAME = "magebane_69"
PASSWORD = "DAD7u3ucwiB6dRnz0aM4ZXu8MKgG3B"
TOPIC = "hello-world"

JAAS_CONFIG = "kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username=\"" + USERNAME + "\" password=\"" + PASSWORD + "\";"

In [0]:
df_raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", JAAS_CONFIG)
    .load()
)

In [0]:
from pyspark.sql.functions import col

df_parsed = df_raw.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

In [0]:
query = (df_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/wh_gb_test/gb_test/my_volume/checkpoints/redpanda-hello-world")
    .toTable("wh_gb_test.gb_test.redpanda_stream")
)

query.awaitTermination()